<a href="https://colab.research.google.com/github/2303A51908/Reinforecement-Learning---B12/blob/main/2303A51908_RL_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

# -----------------------------
# Hyperparameters
# -----------------------------
GAMMA = 0.99            # Discount factor
GAE_LAMBDA = 0.95       # GAE parameter
LR = 3e-4               # Learning rate
EPISODES = 500
STEPS_PER_EPOCH = 200   # Rollout length/steps per update
ENTROPY_BETA = 0.001    # Entropy regularization coefficient
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 1. Actor-Critic Model
# -----------------------------
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, act_limit):
        super().__init__()
        self.act_limit = act_limit

        # Actor: Outputs mean (mu) for Gaussian distribution
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, act_dim)
        )
        self.log_std = nn.Parameter(torch.zeros(act_dim))

        # Critic: Outputs state-value (V)
        self.critic = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, obs):
        # obs shape must be (Batch_Size, obs_dim) e.g., (1, 3) or (200, 3)
        val = self.critic(obs)

        # Actor output: Mean (mu)
        mu = self.actor(obs)
        mu = torch.tanh(mu) * self.act_limit

        std = torch.exp(self.log_std).expand_as(mu)

        return mu, std, val

    def get_action(self, obs):
        """Samples an action and computes log_prob and value."""
        with torch.no_grad():
            mu, std, val = self.forward(obs)
            dist = Normal(mu, std)

            # Sample action
            action = dist.sample()
            logp = dist.log_prob(action).sum(axis=-1)

            # --- FIX: Ensure a_env is a NumPy array of shape (1,) or (1, 1) ---
            # It's safest to convert to numpy and ensure the shape is correct
            a_env = action.cpu().numpy().flatten()

            return a_env, logp, val.squeeze()

# -----------------------------
# 2. GAE and Training Logic
# -----------------------------
def compute_gae(rewards, values, dones, next_val, gamma, gae_lambda):
    """Computes Generalised Advantage Estimation (GAE) and Monte-Carlo returns."""
    advantages = []
    returns = []

    values = values + [next_val]

    gae = 0.0

    for i in reversed(range(len(rewards))):
        delta = rewards[i] + gamma * values[i+1] * (1.0 - dones[i]) - values[i]

        gae = delta + gamma * gae_lambda * (1.0 - dones[i]) * gae
        advantages.insert(0, gae)

        returns.insert(0, gae + values[i])

    return torch.tensor(advantages, dtype=torch.float32, device=DEVICE), \
           torch.tensor(returns, dtype=torch.float32, device=DEVICE)

def train_a2c():
    env = gym.make("Pendulum-v1")

    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    act_limit = env.action_space.high[0]

    model = ActorCritic(obs_dim, act_dim, act_limit).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for ep in range(EPISODES):
        obs, _ = env.reset()
        ep_reward = 0

        obs_buf, act_buf, logp_buf, rew_buf, val_buf, done_buf = [], [], [], [], [], []
        done = False

        # --- A. Collect Rollout ---
        for t in range(STEPS_PER_EPOCH):
            # Input shape MUST be (1, 3). obs is (3,), unsqueeze(0) makes it (1, 3).
            o_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)

            a, logp, val = model.get_action(o_t)

            # Step the environment (a is shape (1,))
            next_obs, r, terminated, truncated, info = env.step(a)
            done = terminated or truncated

            # Store in buffers
            obs_buf.append(obs)
            # --- CRITICAL FIX 2: Store action as 1D array to maintain shape (1,) ---
            act_buf.append(a.astype(np.float32))
            logp_buf.append(logp.item())
            rew_buf.append(r)
            val_buf.append(val.item())
            done_buf.append(float(done))

            obs = next_obs
            ep_reward += r

            if done:
                next_val = 0.0
                obs, _ = env.reset()
                break

        if not done:
            o_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                _, _, next_val_t = model.forward(o_t)
                next_val = next_val_t.squeeze().item()
            obs, _ = env.reset()

        # --- B. Compute Returns and Advantages (GAE) ---
        advs, returns = compute_gae(
            rew_buf, val_buf, done_buf, next_val, GAMMA, GAE_LAMBDA
        )

        # --- C. Update Model ---
        # np.vstack ensures correct stacking: obs_t (T, 3), act_t (T, 1)
        obs_t = torch.tensor(np.vstack(obs_buf), dtype=torch.float32, device=DEVICE)
        act_t = torch.tensor(np.vstack(act_buf), dtype=torch.float32, device=DEVICE)

        mu, std, vals = model(obs_t)
        dist = Normal(mu, std)

        new_logp = dist.log_prob(act_t).sum(1)
        entropy = dist.entropy().sum(1).mean()

        advs = advs.unsqueeze(-1)

        # Loss calculation
        actor_loss = -(new_logp * advs.detach()).mean()
        critic_loss = nn.MSELoss()(vals, returns.unsqueeze(-1))

        loss = actor_loss + 0.5 * critic_loss - ENTROPY_BETA * entropy

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Episode {ep+1}/{EPISODES}, Reward: {ep_reward:.2f}, Length: {len(rew_buf)}")

    env.close()

# -----------------------------
# Run Training
# -----------------------------
if __name__ == "__main__":
    train_a2c()

Episode 1/500, Reward: -1607.00, Length: 200
Episode 2/500, Reward: -1577.45, Length: 200
Episode 3/500, Reward: -1026.52, Length: 200
Episode 4/500, Reward: -875.02, Length: 200
Episode 5/500, Reward: -1328.52, Length: 200
Episode 6/500, Reward: -1654.35, Length: 200
Episode 7/500, Reward: -1670.00, Length: 200
Episode 8/500, Reward: -1547.82, Length: 200
Episode 9/500, Reward: -1335.61, Length: 200
Episode 10/500, Reward: -1469.12, Length: 200
Episode 11/500, Reward: -1845.80, Length: 200
Episode 12/500, Reward: -1510.59, Length: 200
Episode 13/500, Reward: -1453.41, Length: 200
Episode 14/500, Reward: -1435.14, Length: 200
Episode 15/500, Reward: -1434.08, Length: 200
Episode 16/500, Reward: -1478.35, Length: 200
Episode 17/500, Reward: -1520.77, Length: 200
Episode 18/500, Reward: -1645.26, Length: 200
Episode 19/500, Reward: -1724.74, Length: 200
Episode 20/500, Reward: -1475.55, Length: 200
Episode 21/500, Reward: -1521.24, Length: 200
Episode 22/500, Reward: -1290.20, Length: 20